In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
sentence = "Repeat is the best medicine for memory".split()
sentence

['Repeat', 'is', 'the', 'best', 'medicine', 'for', 'memory']

In [4]:
vocab = list(set(sentence))

In [5]:
word2index = {tkn: i for i, tkn in enumerate(vocab, 1)}
word2index['<unk>'] = 0

In [6]:
word2index

{'the': 1,
 'is': 2,
 'best': 3,
 'for': 4,
 'Repeat': 5,
 'memory': 6,
 'medicine': 7,
 '<unk>': 0}

In [8]:
index2word = {v: k for k, v in word2index.items()}
index2word

{1: 'the',
 2: 'is',
 3: 'best',
 4: 'for',
 5: 'Repeat',
 6: 'memory',
 7: 'medicine',
 0: '<unk>'}

In [13]:
def build_data(sentence, word2index):
    encoded = [word2index[token] for token in sentence]
    input_seq, label_seq = encoded[:-1], encoded[1:]
    input_seq = torch.LongTensor(input_seq).unsqueeze(0)
    label_seq = torch.LongTensor(label_seq).unsqueeze(0)

    return input_seq, label_seq

In [14]:
X, Y = build_data(sentence, word2index)

X, Y

(tensor([[5, 2, 1, 3, 7, 4]]), tensor([[2, 1, 3, 7, 4, 6]]))

In [15]:
class Net(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size, batch_first=True):
        super(Net, self).__init__()
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=input_size)
        # 입력 차원, 은닉 상태의 크기 정의
        self.rnn_layer = nn.RNN(input_size, hidden_size, batch_first=batch_first)
        # linear의 output은 원-핫 벡터 즉, 사전의 크기만큼 가져야함
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # 1. 임베딩 층
        # 크기 변화: (배치 크기, 시퀀스 길이) → (배치 크기, 시퀀스 길이, 임베딩 차원) 
        output = self.embedding_layer(x)
        # 2. RNN 층
        # 크기 변화: (배치 크기, 시퀀스 길이, 임베딩 차원) → output(배치 크기, 시퀀스 길이, 은닉층 크기), hidden (1, 배치 크기, 은닉층 크기)
        output, hidden = self.rnn_layer(output)
        # 3. 최종 출력 층
        # 크기 변화: (배치 크기, 시퀀스 길이, 은닉층 크기) → (배치 크기, 시퀀스 길이, 단어장 길이)
        output = self.linear(output)

        # 4. view를 통해서 배치 차원 제거
        # 크기 변화: (배치 크기, 시퀀스 길이, 단어장 크기) → (배치 크기*시퀀스 길이, 단어장 크기)
        return output.view(-1, output.size(2))

In [16]:
vocab_size = len(word2index)
input_size = 5
hidden_size = 20

In [17]:
model = Net(vocab_size, input_size, hidden_size, batch_first=True)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters())

In [18]:
decode = lambda y: [index2word.get(x) for x in y]

In [21]:
for step in range(201):
    optimizer.zero_grad()

    output = model(X)
    loss = loss_function(output, Y.view(-1))
    loss.backward()
    optimizer.step()

    if step==200:
        pred = output.softmax(-1).argmax(-1).tolist()
        print(" ".join(["Repeat"] + decode(pred)))

Repeat is the best medicine for memory
